# SXF — GPU Benchmark in Google Colab

**Runs remotely on Google's GPU, not on your MacBook.** No laptop GPU or model installation is required.

This notebook runs two **real inference trials** using the same checked-in SXF measurement script: (1) `qwen3:8b` chat, (2) `qwen3-vl:4b` with a deterministic synthetic image. For each model: **one warmup + three recorded trials**. It downloads a small ZIP file of raw evidence for human review.

**Before running:** In Colab select **Runtime → Change runtime type → Hardware accelerator → T4 GPU (or another available GPU)**. Free GPU access is not guaranteed by Google; if there is no GPU the notebook stops before downloading models. Do not select any paid upgrade unless you choose to do so. Run **Runtime → Run all**. Both models download to the Colab server, never to your Mac. Runtime GPU and disk availability may vary.

**Evidence limits:** `nvidia-smi` samples total GPU usage, not guaranteed model-exclusive peak VRAM. The notebook does not certify full GPU residence, quality of image answers, multi-GPU behavior, or performance of any other GPU. The public SXF benchmark registry is **never modified automatically**. The image contains a red square on a blue background; its SHA-256 is recorded, but automatic scoring of vision accuracy is not claimed.

In [ ]:
import shutil, subprocess, pathlib
gpu = shutil.which("nvidia-smi")
if not gpu:
    raise RuntimeError("No NVIDIA GPU assigned in Colab. Use Runtime → Change runtime type → GPU, or retry when free GPUs are available.")
check = subprocess.run([gpu, "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, check=True)
rows = [x.strip() for x in check.stdout.splitlines() if x.strip()]
if len(rows) != 1:
    raise RuntimeError("This evidence procedure requires exactly one GPU.")
print("Cloud GPU:", rows[0])
print("Evidence will be collected on the Colab server, not your Mac.")

In [ ]:
import pathlib, subprocess, sys
ROOT = pathlib.Path("/content/sxf-ai")
OUTPUT = pathlib.Path("/content/sxf-gpu-evidence")
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (ROOT / ".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch","main","https://github.com/rab2323s-creator/sxf-ai.git",str(ROOT)],check=True)
else:
    # Colab retains this checkout between notebook runs; get the latest SXF tools.
    # --ff-only refuses to discard divergent local history.
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"], check=True)
if not (ROOT / "scripts" / "ensure_colab_ollama.py").is_file():
    raise RuntimeError("SXF checkout is stale: missing Ollama bootstrap; rerun setup.")
subprocess.run([sys.executable,"scripts/audit_local_vision_sources.py"],cwd=ROOT,check=True)
subprocess.run([sys.executable,"tests/test_local_runtime_evidence.py"],cwd=ROOT,check=True)
print("Repository and source-contract tests ready.")

In [ ]:
import subprocess, sys
# Explicitly synchronize Colab's reused clone and reload Python's cached module.
subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "FETCH_HEAD"], check=True)
import importlib
sys.path.insert(0, str(ROOT / "scripts"))
import ensure_colab_ollama
importlib.reload(ensure_colab_ollama)
print("Ollama bootstrap source:", ensure_colab_ollama.__file__)
from ensure_colab_ollama import ensure_ollama_ready
OLLAMA = ensure_ollama_ready(OUTPUT)
print("Ollama installation and local server confirmed.")

## Text benchmark — Qwen3 8B

This model is a *screening control*, not a claim about every 8B model. The data is never published without manual review.

In [ ]:
import subprocess, sys
# Explicitly synchronize Colab's reused clone and reload Python's cached module.
subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "FETCH_HEAD"], check=True)
import importlib
sys.path.insert(0, str(ROOT / "scripts"))
import ensure_colab_ollama
importlib.reload(ensure_colab_ollama)
print("Ollama bootstrap source:", ensure_colab_ollama.__file__)
from ensure_colab_ollama import ensure_ollama_ready
OLLAMA = ensure_ollama_ready(OUTPUT)
text_result = OUTPUT / "qwen3-8b-4096.json"
if text_result.exists():
    print("Existing text measurement preserved:", text_result)
else:
    subprocess.run([OLLAMA, "pull", "qwen3:8b"], check=True)
    print("Running real Ollama inference; errors below are from measurement script.", flush=True)
    run = subprocess.run([sys.executable, "-u", "scripts/measure_local_ollama.py",
                    "--model-id", "ollama:qwen3:8b", "--context", "4096", "--repeats", "3",
                    "--output", str(text_result)], cwd=ROOT, capture_output=True, text=True)
    if run.stdout: print(run.stdout[-12000:], flush=True)
    if run.stderr: print(run.stderr[-12000:], flush=True)
    if run.returncode: raise RuntimeError("Measurement failed (exit " + str(run.returncode) + "); see original diagnostic above.")
print("Text run saved:", text_result)

## Vision benchmark — Qwen3-VL 4B

A deterministic local PNG is generated on Colab, then passed to the actual Ollama image API. This validates the **request path and raw speed telemetry**, not the correctness of image understanding. Its SHA-256 is stored by the measurement script.

In [ ]:
import struct, zlib, hashlib, subprocess, sys
# Explicitly synchronize Colab's reused clone and reload Python's cached module.
subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "FETCH_HEAD"], check=True)
import importlib
sys.path.insert(0, str(ROOT / "scripts"))
import ensure_colab_ollama
importlib.reload(ensure_colab_ollama)
print("Ollama bootstrap source:", ensure_colab_ollama.__file__)
from ensure_colab_ollama import ensure_ollama_ready
w,h=128,128
raw=b"".join(b"\x00"+b"".join(bytes((238,42,42)) if 24<=x<104 and 24<=y<104 else bytes((36,77,208)) for x in range(w)) for y in range(h))
def pngchunk(tag,data):
    return struct.pack("!I",len(data))+tag+data+struct.pack("!I",zlib.crc32(tag+data)&0xffffffff)
png=b"\x89PNG\r\n\x1a\n"+pngchunk(b"IHDR",struct.pack("!2I5B",w,h,8,2,0,0,0))+pngchunk(b"IDAT",zlib.compress(raw))+pngchunk(b"IEND",b"")
image_file=OUTPUT/"vision-square.png"
image_file.write_bytes(png)
print("Vision fixture:",str(image_file),"sha256:",hashlib.sha256(png).hexdigest())
OLLAMA = ensure_ollama_ready(OUTPUT)
if not (OUTPUT / "qwen3-vl-4b-4096.json").exists():
    subprocess.run([OLLAMA,"pull","qwen3-vl:4b"],check=True)
vision_result=OUTPUT/"qwen3-vl-4b-4096.json"
if not vision_result.exists():
    print("Running real Ollama inference; errors below are from measurement script.", flush=True)
    run = subprocess.run([sys.executable,"-u","scripts/measure_local_ollama.py",
                "--model-id","ollama:qwen3-vl:4b","--context","4096","--repeats","3",
                "--image",str(image_file),"--output",str(vision_result)], cwd=ROOT, capture_output=True, text=True)
    if run.stdout: print(run.stdout[-12000:], flush=True)
    if run.stderr: print(run.stderr[-12000:], flush=True)
    if run.returncode: raise RuntimeError("Measurement failed (exit " + str(run.returncode) + "); see original diagnostic above.")
print("Vision run saved:",vision_result)

## Export raw evidence

Download the ZIP and attach it to the ChatGPT conversation for review. **Never label the runs "verified compatible" without checking full GPU residency, model version, and image-output validity.** The ZIP includes raw JSON and the synthetic image; no personal image is used.

In [ ]:
import datetime, hashlib, json, zipfile
from google.colab import files
reports=sorted(OUTPUT.glob("*.json"))
if not reports:
    raise RuntimeError("No measured evidence. A GPU run must succeed before export.")
manifest={"source":"SXF cloud notebook","recordedAtUtc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
          "publicationStatus":"review-required","files":[]}
for report in reports:
    data=json.loads(report.read_text())
    if data.get("kind")!="local-measurement-candidate" or data.get("publicationStatus")!="review-required":
        raise RuntimeError("Unexpected evidence format: "+report.name)
    manifest["files"].append({"name":report.name,"sha256":hashlib.sha256(report.read_bytes()).hexdigest(),
                              "modelId":data["modelId"],"gpuName":data["gpu"]["name"]})
manifest_file=OUTPUT/"manifest.json"
manifest_file.write_text(json.dumps(manifest,indent=2)+"\n")
archive=pathlib.Path("/content/sxf-cloud-gpu-evidence.zip")
if archive.exists(): archive.unlink()
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for name in [*reports,manifest_file,OUTPUT/"vision-square.png"]:
        if name.exists():z.write(name,name.name)
print("Prepared",archive,"with",len(reports),"raw run files. Results have NOT been published.")
files.download(str(archive))